# Fusformer Kaggle Test Notebook

This notebook runs the pretrained Fusformer checkpoint on the CAVE test scenes stored as HSI and RGB MAT files.

It keeps only the test-time flow: load the checkpoint, load the scene pairs, build the model input, run inference, and save the outputs.

In [11]:
import os
import glob
import math
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from torch.utils.data import Dataset, DataLoader

SEED = 1
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CHECKPOINT_PATH = '/kaggle/input/models/nikeshreddypatlolla/model-fusformer/pytorch/default/1/1000.pth'
HSI_DIR = '/kaggle/input/datasets/nikeshreddypatlolla/cave-dataset-2/Data/Test/HSI'
RGB_DIR = '/kaggle/input/datasets/nikeshreddypatlolla/cave-dataset-2/Data/Test/RGB'
TEST_TXT = '/kaggle/input/datasets/nikeshreddypatlolla/cave-dataset-2/Data/Test/Test.txt'
OUTPUT_DIR = '/kaggle/working/fusformer_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('device:', device)
print('checkpoint exists:', os.path.exists(CHECKPOINT_PATH))
print('hsi dir exists:', os.path.exists(HSI_DIR))
print('rgb dir exists:', os.path.exists(RGB_DIR))

device: cuda
checkpoint exists: True
hsi dir exists: True
rgb dir exists: True


In [12]:
# Copied from model.py

def init_weights(*modules):
    for module in modules:
        for m in module.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_in')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1.0)
                nn.init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.0)


class MainNet(nn.Module):

    def __init__(self):
        super(MainNet, self).__init__()
        num_channel = 31
        num_feature = 48
        self.T_E = Transformer_E(num_feature)
        self.T_D = Transformer_D(num_feature)
        self.Embedding = nn.Sequential(
            nn.Linear(num_channel + 3, num_feature),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(num_feature, num_feature, 3, 1, 1),
            nn.LeakyReLU(),
            nn.Conv2d(num_feature, num_channel, 3, 1, 1)
        )

    def forward(self, HSI, MSI):
        UP_LRHSI = F.interpolate(HSI, scale_factor=4, mode='bicubic')
        UP_LRHSI = UP_LRHSI.clamp_(0, 1)
        sz = UP_LRHSI.size(2)
        Data = torch.cat((UP_LRHSI, MSI), 1)
        E = rearrange(Data, 'B c H W -> B (H W) c', H=sz)
        E = self.Embedding(E)
        Code = self.T_E(E)
        Highpass = self.T_D(Code)
        Highpass = rearrange(Highpass, 'B (H W) C -> B C H W', H=sz)
        Highpass = self.refine(Highpass)
        output = Highpass + UP_LRHSI
        output = output.clamp_(0, 1)

        return output, UP_LRHSI, Highpass


class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x


class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(self.norm(x), **kwargs)


class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.LeakyReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Attention(nn.Module):
    def __init__(self, dim, heads, dim_head, dropout=0.):
        super().__init__()
        inner_dim = dim_head * heads
        project_out = not (heads == 1 and dim_head == dim)

        self.heads = heads
        self.scale = dim_head ** -0.5
        self.to_qkv = nn.Linear(dim, inner_dim * 3, bias=False)
        self.to_out = nn.Sequential(
            nn.Linear(inner_dim, dim),
            nn.Dropout(dropout)
        ) if project_out else nn.Identity()

    def forward(self, x, mask=None):
        b, n, _, h = *x.shape, self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)

        dots = torch.einsum('b h i d, b h j d -> b h i j', q, k) * self.scale
        mask_value = -torch.finfo(dots.dtype).max

        if mask is not None:
            mask = F.pad(mask.flatten(1), (1, 0), value=True)
            assert mask.shape[-1] == dots.shape[-1], 'mask has incorrect dimensions'
            mask = rearrange(mask, 'b i -> b () i ()') * rearrange(mask, 'b j -> b () () j')
            dots.masked_fill_(~mask, mask_value)
            del mask

        attn = dots.softmax(dim=-1)
        out = torch.einsum('b h i j, b h j d -> b h i d', attn, v)
        out = rearrange(out, 'b h n d -> b n (h d)')
        out = self.to_out(out)
        return out


class Transformer_E(nn.Module):
    def __init__(self, dim, depth=2, heads=3, dim_head=16, mlp_dim=48, sp_sz=64*64, num_channels=48, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        self.pos_embedding = nn.Parameter(torch.randn(1, sp_sz, num_channels))
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Residual(PreNorm(dim, Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout))),
                Residual(PreNorm(dim, FeedForward(dim, mlp_dim, dropout=dropout)))
            ]))

    def forward(self, x, mask=None):
        for attn, ff in self.layers:
            x = attn(x, mask=mask)
            x = ff(x)
        return x


class Transformer_D(nn.Module):
    def __init__(self, dim, depth=2, heads=3, dim_head=16, mlp_dim=48, sp_sz=64*64, num_channels=48, dropout=0.):
        super().__init__()
        self.layers = nn.ModuleList([])
        self.pos_embedding = nn.Parameter(torch.randn(1, sp_sz, num_channels))
        for _ in range(depth):
            self.layers.append(nn.ModuleList([
                Residual(PreNorm(dim, Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout))),
                Residual(PreNorm(dim, Attention(dim, heads=heads, dim_head=dim_head, dropout=dropout))),
                Residual(PreNorm(dim, FeedForward(dim, mlp_dim, dropout=dropout)))
            ]))

    def forward(self, x, mask=None):
        for attn1, attn2, ff in self.layers:
            x = attn1(x, mask=mask)
            x = attn2(x, mask=mask)
            x = ff(x)
        return x

In [13]:
def _first_data_key(mat_obj):
    for key in mat_obj.keys():
        if not key.startswith('__'):
            return key
    raise ValueError('No data array found in MAT file')


def load_mat_array(path, key=None):
    mat_obj = sio.loadmat(path)
    if key is None:
        key = _first_data_key(mat_obj)
    if key not in mat_obj:
        raise KeyError(f'Key {key} not found in {path}')
    return np.asarray(mat_obj[key])


def _to_nchw(arr, channels, name):
    arr = np.asarray(arr)
    if arr.ndim == 3:
        if arr.shape[0] == channels:
            arr = arr[None, ...]
        elif arr.shape[-1] == channels:
            arr = np.transpose(arr, (2, 0, 1))[None, ...]
        else:
            raise ValueError(f'{name} shape {arr.shape} does not match {channels} channels')
    elif arr.ndim == 4:
        if arr.shape[1] == channels:
            pass
        elif arr.shape[-1] == channels:
            arr = np.transpose(arr, (0, 3, 1, 2))
        else:
            raise ValueError(f'{name} shape {arr.shape} does not match {channels} channels')
    else:
        raise ValueError(f'{name} must be 3D or 4D, got {arr.shape}')
    return arr.astype(np.float32)


def read_scene_list(test_txt_path):
    if test_txt_path and os.path.exists(test_txt_path):
        with open(test_txt_path, 'r') as f:
            names = [line.strip() for line in f.readlines() if line.strip()]
        return names
    return None


def find_mat_by_stem(folder, stem):
    candidates = sorted(glob.glob(os.path.join(folder, stem + '.mat')))
    if candidates:
        return candidates[0]
    candidates = sorted(glob.glob(os.path.join(folder, stem + '*.mat')))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f'No MAT file found for {stem} in {folder}')


def build_scene_pairs(hsi_dir, rgb_dir, test_txt_path=None):
    names = read_scene_list(test_txt_path)
    if names is None:
        hsi_stems = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(hsi_dir, '*.mat'))}
        rgb_stems = {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(rgb_dir, '*.mat'))}
        names = sorted(list(hsi_stems & rgb_stems))
    pairs = []
    for name in names:
        hsi_path = find_mat_by_stem(hsi_dir, name)
        rgb_path = find_mat_by_stem(rgb_dir, name)
        pairs.append((name, hsi_path, rgb_path))
    return pairs


class CaveSceneTestDataset(Dataset):
    def __init__(self, hsi_dir, rgb_dir, test_txt_path=None, hsi_key=None, rgb_key=None):
        self.pairs = build_scene_pairs(hsi_dir, rgb_dir, test_txt_path)
        self.hsi_key = hsi_key
        self.rgb_key = rgb_key

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        scene_name, hsi_path, rgb_path = self.pairs[index]
        hsi = load_mat_array(hsi_path, self.hsi_key)
        rgb = load_mat_array(rgb_path, self.rgb_key)
        hsi = _to_nchw(hsi, 31, 'HSI')[0]
        rgb = _to_nchw(rgb, 3, 'RGB')[0]
        hsi_t = torch.from_numpy(hsi).float()
        rgb_t = torch.from_numpy(rgb).float()
        if rgb_t.shape[-2:] != hsi_t.shape[-2:]:
            rgb_t = F.interpolate(rgb_t.unsqueeze(0), size=hsi_t.shape[-2:], mode='bicubic', align_corners=False).squeeze(0)
        lr_size = (max(1, hsi_t.shape[-2] // 4), max(1, hsi_t.shape[-1] // 4))
        lrhsi_t = F.interpolate(hsi_t.unsqueeze(0), size=lr_size, mode='bicubic', align_corners=False).squeeze(0)
        return scene_name, hsi_t, lrhsi_t, rgb_t


def load_checkpoint_robust(model, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint['net'] if isinstance(checkpoint, dict) and 'net' in checkpoint else checkpoint
    try:
        model.load_state_dict(state_dict)
        return
    except RuntimeError:
        pass

    if isinstance(model, nn.DataParallel):
        normalized = {}
        for key, value in state_dict.items():
            if not key.startswith('module.'):
                normalized['module.' + key] = value
            else:
                normalized[key] = value
        model.load_state_dict(normalized)
    else:
        normalized = {}
        for key, value in state_dict.items():
            if key.startswith('module.'):
                normalized[key[len('module.'):]] = value
            else:
                normalized[key] = value
        model.load_state_dict(normalized)


print('helper functions ready')

helper functions ready


In [14]:
def psnr(pred, ref, data_range=1.0):
    mse = np.mean((pred - ref) ** 2)
    if mse <= 1e-12:
        return 99.0
    return 10.0 * np.log10((data_range ** 2) / mse)


def sam(pred, ref, eps=1e-12):
    p = pred.reshape(-1, pred.shape[-1])
    r = ref.reshape(-1, ref.shape[-1])
    dot = np.sum(p * r, axis=1)
    pn = np.linalg.norm(p, axis=1)
    rn = np.linalg.norm(r, axis=1)
    cosang = dot / np.maximum(pn * rn, eps)
    cosang = np.clip(cosang, -1.0, 1.0)
    return float(np.mean(np.degrees(np.arccos(cosang))))


def ergas(pred, ref, scale=4, eps=1e-12):
    # Standard ERGAS formula using per-band RMSE and mean reference band value.
    bands = ref.shape[-1]
    rmse_b = np.sqrt(np.mean((pred - ref) ** 2, axis=(0, 1)))
    mean_ref_b = np.mean(ref, axis=(0, 1))
    ratio_sq = (rmse_b / np.maximum(mean_ref_b, eps)) ** 2
    return float((100.0 / scale) * np.sqrt(np.mean(ratio_sq)))


def ssim_per_band(pred, ref, data_range=1.0, k1=0.01, k2=0.03, eps=1e-12):
    c1 = (k1 * data_range) ** 2
    c2 = (k2 * data_range) ** 2
    vals = []
    for b in range(ref.shape[-1]):
        x = pred[:, :, b]
        y = ref[:, :, b]
        mux = np.mean(x)
        muy = np.mean(y)
        sigx = np.mean((x - mux) ** 2)
        sigy = np.mean((y - muy) ** 2)
        sigxy = np.mean((x - mux) * (y - muy))
        num = (2 * mux * muy + c1) * (2 * sigxy + c2)
        den = (mux * mux + muy * muy + c1) * (sigx + sigy + c2)
        vals.append(float(num / max(den, eps)))
    return float(np.mean(vals))


print('metric helpers ready')

metric helpers ready


In [15]:
# ---------------------------------------------------------------
# patch_infer: tile-based inference to avoid OOM.
# The model was trained on 64x64 HR patches (16x16 LR patches).
# Passing the full image produces a 65536-token sequence whose
# attention matrix would need ~768 GiB. Instead we split the
# HR image into non-overlapping 64x64 tiles, run the unchanged
# model on each tile, then stitch the outputs back together.
# Model classes (MainNet, Transformer_E, Transformer_D, etc.)
# are NOT modified at all.
# ---------------------------------------------------------------

PATCH_HR = 64   # HR patch size the model was trained on
PATCH_LR = PATCH_HR // 4  # corresponding LR patch size


def patch_infer(model, lrhsi, rgb, patch_hr=64):
    """
    Run the model tile-by-tile over a full image.

    Args:
        model  : the unchanged MainNet (or DataParallel wrapper)
        lrhsi  : (1, C, H_lr, W_lr) LR HSI tensor on device
        rgb    : (1, 3, H_hr, W_hr) HR RGB tensor on device
        patch_hr: HR patch size (default 64, matching training)
    Returns:
        output, up_lrhsi, highpass  each (1, C, H_hr, W_hr)
    """
    patch_lr = patch_hr // 4
    _, C_hsi, H_lr, W_lr = lrhsi.shape
    _, C_rgb, H_hr, W_hr = rgb.shape

    # Pad so that dimensions are multiples of patch_lr / patch_hr
    pad_h_lr = (patch_lr - H_lr % patch_lr) % patch_lr
    pad_w_lr = (patch_lr - W_lr % patch_lr) % patch_lr
    lrhsi_pad = F.pad(lrhsi, (0, pad_w_lr, 0, pad_h_lr), mode='reflect')
    rgb_pad   = F.pad(rgb,   (0, pad_w_lr * 4, 0, pad_h_lr * 4), mode='reflect')

    _, _, H_lr_p, W_lr_p = lrhsi_pad.shape
    H_hr_p, W_hr_p = H_lr_p * 4, W_lr_p * 4

    out_buf  = torch.zeros(1, C_hsi, H_hr_p, W_hr_p, device=lrhsi.device)
    up_buf   = torch.zeros(1, C_hsi, H_hr_p, W_hr_p, device=lrhsi.device)
    high_buf = torch.zeros(1, C_hsi, H_hr_p, W_hr_p, device=lrhsi.device)

    n_h = H_lr_p // patch_lr
    n_w = W_lr_p // patch_lr

    for i in range(n_h):
        for j in range(n_w):
            lr_r0, lr_r1 = i * patch_lr, (i + 1) * patch_lr
            lr_c0, lr_c1 = j * patch_lr, (j + 1) * patch_lr
            hr_r0, hr_r1 = i * patch_hr, (i + 1) * patch_hr
            hr_c0, hr_c1 = j * patch_hr, (j + 1) * patch_hr

            lr_tile  = lrhsi_pad[:, :, lr_r0:lr_r1, lr_c0:lr_c1]
            rgb_tile = rgb_pad[:,   :, hr_r0:hr_r1, hr_c0:hr_c1]

            out_t, up_t, high_t = model(lr_tile, rgb_tile)

            out_buf[:,  :, hr_r0:hr_r1, hr_c0:hr_c1] = out_t
            up_buf[:,   :, hr_r0:hr_r1, hr_c0:hr_c1] = up_t
            high_buf[:, :, hr_r0:hr_r1, hr_c0:hr_c1] = high_t

    # Crop padding back to original HR size
    return (
        out_buf[:,  :, :H_hr, :W_hr],
        up_buf[:,   :, :H_hr, :W_hr],
        high_buf[:, :, :H_hr, :W_hr],
    )


test_dataset = CaveSceneTestDataset(
    hsi_dir=HSI_DIR,
    rgb_dir=RGB_DIR,
    test_txt_path=TEST_TXT,
)

print('number of test scenes:', len(test_dataset))

model = MainNet().to(device)
# NOTE: skip DataParallel – single-GPU tile inference is sufficient
# and DataParallel would try to scatter the full image across GPUs.

load_checkpoint_robust(model, CHECKPOINT_PATH)
model.eval()
print('checkpoint loaded')

saved_files = []
psnr_scores = []
sam_scores = []
ergas_scores = []
ssim_scores = []

with torch.no_grad():
    for scene_name, hsi, lrhsi, rgb in test_dataset:
        hsi    = hsi.unsqueeze(0).to(device)
        lrhsi  = lrhsi.unsqueeze(0).to(device)
        rgb    = rgb.unsqueeze(0).to(device)

        # Use patch_infer instead of model(lrhsi, rgb) directly
        # to stay within GPU memory (64x64 tiles = 4096 tokens).
        output, up_lrhsi, highpass = patch_infer(model, lrhsi, rgb, patch_hr=PATCH_HR)

        output_np = output.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)
        gt_np     = hsi.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)
        up_np     = up_lrhsi.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)
        high_np   = highpass.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)

        scene_psnr  = psnr(output_np, gt_np)
        scene_sam   = sam(output_np, gt_np)
        scene_ergas = ergas(output_np, gt_np, scale=4)
        scene_ssim  = ssim_per_band(output_np, gt_np)

        psnr_scores.append(scene_psnr)
        sam_scores.append(scene_sam)
        ergas_scores.append(scene_ergas)
        ssim_scores.append(scene_ssim)

        out_path = os.path.join(OUTPUT_DIR, f'{scene_name}_output.mat')
        sio.savemat(out_path, {
            'output': output_np,
            'gt': gt_np,
            'up_lrhsi': up_np,
            'highpass': high_np,
        })
        saved_files.append(out_path)
        print(
            f'{scene_name}: PSNR={scene_psnr:.4f} SAM={scene_sam:.4f} '
            f'ERGAS={scene_ergas:.4f} SSIM={scene_ssim:.4f} -> {out_path}'
        )
        torch.cuda.empty_cache()



# ── Results Table ──────────────────────────────────────────────────
col_w = [30, 10, 10, 10, 10]  # column widths
headers = ['Scene', 'PSNR', 'SAM', 'ERGAS', 'SSIM']
divider = '+' + '+'.join('-' * w for w in col_w) + '+'
fmt_row = lambda vals: '|' + '|'.join(str(v).center(w) for v, w in zip(vals, col_w)) + '|'

print(divider)
print(fmt_row(headers))
print(divider.replace('-', '='))

scene_names_list = [pair[0] for pair in test_dataset.pairs]
for name, p, s, e, ss in zip(scene_names_list, psnr_scores, sam_scores, ergas_scores, ssim_scores):
    print(fmt_row([name[:28], f'{p:.4f}', f'{s:.4f}', f'{e:.4f}', f'{ss:.4f}']))
    print(divider)

# Average row
avg_vals = [
    'AVERAGE',
    f'{np.mean(psnr_scores):.4f}',
    f'{np.mean(sam_scores):.4f}',
    f'{np.mean(ergas_scores):.4f}',
    f'{np.mean(ssim_scores):.4f}',
]
print(fmt_row(avg_vals))
print(divider)

print(f'\nTotal scenes evaluated : {len(saved_files)}')
print(f'Output directory       : {OUTPUT_DIR}')


number of test scenes: 12
checkpoint loaded
superballs: PSNR=52.5093 SAM=3.1449 ERGAS=1.2704 SSIM=0.9994 -> /kaggle/working/fusformer_outputs/superballs_output.mat
thread_spools: PSNR=51.2675 SAM=2.6574 ERGAS=0.8219 SSIM=0.9996 -> /kaggle/working/fusformer_outputs/thread_spools_output.mat
oil_painting: PSNR=49.5952 SAM=2.3469 ERGAS=0.7988 SSIM=0.9995 -> /kaggle/working/fusformer_outputs/oil_painting_output.mat
paints: PSNR=47.2674 SAM=1.9741 ERGAS=0.5861 SSIM=0.9997 -> /kaggle/working/fusformer_outputs/paints_output.mat
photo_and_face: PSNR=46.0592 SAM=4.1418 ERGAS=2.2893 SSIM=0.9986 -> /kaggle/working/fusformer_outputs/photo_and_face_output.mat
pompoms: PSNR=51.0120 SAM=1.3416 ERGAS=0.4525 SSIM=0.9997 -> /kaggle/working/fusformer_outputs/pompoms_output.mat
real_and_fake_apples: PSNR=56.3973 SAM=2.4403 ERGAS=0.9003 SSIM=0.9998 -> /kaggle/working/fusformer_outputs/real_and_fake_apples_output.mat
real_and_fake_peppers: PSNR=55.0866 SAM=1.8899 ERGAS=0.5684 SSIM=0.9998 -> /kaggle/working/f